In [141]:
# ==========================================
# 1. IMPORT LIBRARIES
# ==========================================

import pandas as pd
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt
import ast

from sklearn.metrics.pairwise import cosine_similarity

pd.set_option('display.max_columns', None)
pd.set_option('display.max_colwidth', 100)


# ==========================================
# 2. LOAD DATASETS
# ==========================================

# Load cleaned job dataset
jobs = pd.read_csv(
    "../data/processed/jobs_cleaned.csv"
)

# Load user interaction dataset
interactions = pd.read_csv(
    "../data/processed/user_interactions_v2.csv"
)

# Load combined content similarity matrix
combined_similarity = np.load(
    "../models/combined_similarity.npy"
)


# ==========================================
# 3. PREPARE DATA
# ==========================================

# Ensure IDs have consistent data types
jobs['id'] = jobs['id'].astype(int)

interactions['job_id'] = (
    interactions['job_id'].astype(int)
)

# Create job ID to matrix index mapping
job_id_to_index = {
    job_id: index
    for index, job_id in enumerate(jobs['id'])
}


# ==========================================
# 4. VERIFY LOADED DATA
# ==========================================

print("DATA LOADED SUCCESSFULLY")
print("=" * 50)

print("Jobs dataset:", jobs.shape)
print("Interactions dataset:", interactions.shape)
print("Content similarity matrix:", combined_similarity.shape)

print("\nUnique users:", interactions['user_id'].nunique())
print("Unique jobs:", jobs['id'].nunique())
print("Total interactions:", len(interactions))

DATA LOADED SUCCESSFULLY
Jobs dataset: (1020, 13)
Interactions dataset: (80224, 6)
Content similarity matrix: (1020, 1020)

Unique users: 4000
Unique jobs: 1020
Total interactions: 80224


In [142]:
# ==========================================
# DATA VALIDATION
# ==========================================

print("Checking missing values:")
print(interactions.isnull().sum())

print("\nInteraction distribution:")
print(interactions['interaction_type'].value_counts())

print("\nRating distribution:")
print(interactions['rating'].value_counts().sort_index())

print("\nDuplicate interactions:")
print(
    interactions.duplicated(
        subset=['user_id', 'job_id']
    ).sum()
)

print("\nChecking similarity matrix:")
print("Correct dimensions:",
      combined_similarity.shape == (len(jobs), len(jobs)))

print("All similarity values finite:",
      np.isfinite(combined_similarity).all())

Checking missing values:
user_id             0
job_id              0
interaction_type    0
rating              0
relevance_score     0
is_relevant         0
dtype: int64

Interaction distribution:
interaction_type
save     33803
view     32650
apply    13771
Name: count, dtype: int64

Rating distribution:
rating
1    32650
3    33803
5    13771
Name: count, dtype: int64

Duplicate interactions:
0

Checking similarity matrix:
Correct dimensions: True
All similarity values finite: True


**Split interactions into training and testing sets**

In [143]:
# Split interactions: 80% training, 20% testing

train_indices = (
    interactions.groupby('user_id')
    .sample(frac=0.80, random_state=42)
    .index
)

train_interactions = interactions.loc[train_indices].copy()
test_indices = interactions.index.difference(train_indices)
test_interactions = interactions.loc[test_indices].copy()

# Reset indexes
train_interactions = train_interactions.reset_index(drop=True)
test_interactions = test_interactions.reset_index(drop=True)

# Display split results
print("TRAINING AND TESTING SPLIT")
print("=" * 50)

print("Total interactions:", len(interactions))
print("Training interactions:", len(train_interactions))
print("Testing interactions:", len(test_interactions))

print("\nUnique users in training:", train_interactions['user_id'].nunique())
print("Unique users in testing:", test_interactions['user_id'].nunique())

print("\nInteractions per user in training:")
print(train_interactions.groupby('user_id').size().value_counts())

print("\nInteractions per user in testing:")
print(test_interactions.groupby('user_id').size().value_counts())

TRAINING AND TESTING SPLIT
Total interactions: 80224
Training interactions: 64198
Testing interactions: 16026

Unique users in training: 4000
Unique users in testing: 4000

Interactions per user in training:
22    459
10    390
18    265
21    248
14    213
20    211
19    199
23    195
9     173
6     165
12    158
11    156
24    151
7     150
8     148
13    129
25    115
17    110
26    100
15     98
16     90
5      35
4      12
27     11
3       9
28      5
29      4
2       1
Name: count, dtype: int64

Interactions per user in testing:
5    1055
2     761
3     761
6     744
4     501
1     128
7      50
Name: count, dtype: int64


In [144]:
# Validate training and testing split

print("SPLIT VALIDATION")
print("=" * 50)

print("Training percentage:",
      round(len(train_interactions) / len(interactions) * 100, 2), "%")

print("Testing percentage:",
      round(len(test_interactions) / len(interactions) * 100, 2), "%")

print("Users missing from training:",
      len(set(interactions['user_id']) - set(train_interactions['user_id'])))

print("Users missing from testing:",
      len(set(interactions['user_id']) - set(test_interactions['user_id'])))

print(
    "Overlapping interaction records:",
    len(set(train_indices).intersection(set(test_indices)))
)

SPLIT VALIDATION
Training percentage: 80.02 %
Testing percentage: 19.98 %
Users missing from training: 0
Users missing from testing: 0
Overlapping interaction records: 0


**Build the training user–job matrix**

In [145]:
# Build user-job matrix using training data only

user_job_matrix = train_interactions.pivot_table(
    index='user_id',
    columns='job_id',
    values='rating',
    fill_value=0
)

# Calculate user similarity
user_similarity = cosine_similarity(user_job_matrix)

user_similarity_df = pd.DataFrame(
    user_similarity,
    index=user_job_matrix.index,
    columns=user_job_matrix.index
)

# Display results
print("TRAINING USER-JOB MATRIX")
print("=" * 50)

print("Matrix dimensions:", user_job_matrix.shape)
print("Number of users:", user_job_matrix.shape[0])
print("Number of jobs with training interactions:", user_job_matrix.shape[1])

print("\nUser similarity matrix:", user_similarity.shape)

print("\nSample user-job matrix:")
display(user_job_matrix.head())

print("\nSample user similarity matrix:")
display(user_similarity_df.iloc[:5, :5])

TRAINING USER-JOB MATRIX
Matrix dimensions: (4000, 1020)
Number of users: 4000
Number of jobs with training interactions: 1020

User similarity matrix: (4000, 4000)

Sample user-job matrix:


job_id,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26,27,28,29,30,31,32,33,34,35,36,37,38,39,40,41,42,43,44,45,46,47,48,49,50,51,52,53,54,55,56,57,58,59,60,61,62,63,64,65,66,67,68,69,70,71,72,73,74,75,76,77,78,79,80,81,82,83,84,85,86,87,88,89,90,91,92,93,94,95,96,97,98,99,100,101,102,103,104,105,106,107,108,109,110,111,112,113,114,115,116,117,118,119,120,121,122,123,124,125,126,127,128,129,130,131,132,133,134,135,136,137,138,139,140,141,142,143,144,145,146,147,148,149,150,151,152,153,154,155,156,157,158,159,160,161,162,163,164,165,166,167,168,169,170,171,172,173,174,175,176,177,178,179,180,181,182,183,184,185,186,187,188,189,190,191,192,193,194,195,196,197,198,199,200,201,202,203,204,205,206,207,208,209,210,211,212,213,214,215,216,217,218,219,220,221,222,223,224,225,226,227,228,229,230,231,232,233,234,235,236,237,238,239,240,241,242,243,244,245,246,247,248,249,250,251,252,253,254,255,256,257,258,259,260,261,262,263,264,265,266,267,268,269,270,271,272,273,274,275,276,277,278,279,280,281,282,283,284,285,286,287,288,289,290,291,292,293,294,295,296,297,298,299,300,301,302,303,304,305,306,307,308,309,310,311,312,313,314,315,316,317,318,319,320,321,322,323,324,325,326,327,328,329,330,331,332,333,334,335,336,337,338,339,340,341,342,343,344,345,346,347,348,349,350,351,352,353,354,355,356,357,358,359,360,361,362,363,364,365,366,367,368,369,370,371,372,373,374,375,376,377,378,379,380,381,382,383,384,385,386,387,388,389,390,391,392,393,394,395,396,397,398,399,400,401,402,403,404,405,406,407,408,409,410,411,412,413,414,415,416,417,418,419,420,421,422,423,424,425,426,427,428,429,430,431,432,433,434,435,436,437,438,439,440,441,442,443,444,445,446,447,448,449,450,451,452,453,454,455,456,457,458,459,460,461,462,463,464,465,466,467,468,469,470,471,472,473,474,475,476,477,478,479,480,481,482,483,484,485,486,487,488,489,490,491,492,493,494,495,496,497,498,499,500,501,502,503,504,505,506,507,508,509,510,511,512,513,514,515,516,517,518,519,520,521,522,523,524,525,526,527,528,529,530,531,532,533,534,535,536,537,538,539,540,541,542,543,544,545,546,547,548,549,550,551,552,553,554,555,556,557,558,559,560,561,562,563,564,565,566,567,568,569,570,571,572,573,574,575,576,577,578,579,580,581,582,583,584,585,586,587,588,589,590,591,592,593,594,595,596,597,598,599,600,601,602,603,604,605,606,607,608,609,610,611,612,613,614,615,616,617,618,619,620,621,622,623,624,625,626,627,628,629,630,631,632,633,634,635,636,637,638,639,640,641,642,643,644,645,646,647,648,649,650,651,652,653,654,655,656,657,658,659,660,661,662,663,664,665,666,667,668,669,670,671,672,673,674,675,676,677,678,679,680,681,682,683,684,685,686,687,688,689,690,691,692,693,694,695,696,697,698,699,700,701,702,703,704,705,706,707,708,709,710,711,712,713,714,715,716,717,718,719,720,721,722,723,724,725,726,727,728,729,730,731,732,733,734,735,736,737,738,739,740,741,742,743,744,745,746,747,748,749,750,751,752,753,754,755,756,757,758,759,760,761,762,763,764,765,766,767,768,769,770,771,772,773,774,775,776,777,778,779,780,781,782,783,784,785,786,787,788,789,790,791,792,793,794,795,796,797,798,799,800,801,802,803,804,805,806,807,808,809,810,811,812,813,814,815,816,817,818,819,820,821,822,823,824,825,826,827,828,829,830,831,832,833,834,835,836,837,838,839,840,841,842,843,844,845,846,847,848,849,850,851,852,853,854,855,856,857,858,859,860,861,862,863,864,865,866,867,868,869,870,871,872,873,874,875,876,877,878,879,880,881,882,883,884,885,886,887,888,889,890,891,892,893,894,895,896,897,898,899,900,901,902,903,904,905,906,907,908,909,910,911,912,913,914,915,916,917,918,919,920,921,922,923,924,925,926,927,928,929,930,931,932,933,934,935,936,937,938,939,940,941,942,943,944,945,946,947,948,949,950,951,952,953,954,955,956,957,958,959,960,961,962,963,964,965,966,967,968,969,970,971,972,973,974,975,976,977,978,979,980,981,982,983,984,985,986,987,988,989,990,991,992,993,994,995,996,997,998,999,1000,1001,1002,1003,1004,1005,1006,1007,1008,1009,1010,1011,1012,1013,1014,1015,1016,1017,1018,1019,1020



Sample user similarity matrix:


user_id,USER_001,USER_002,USER_003,USER_004,USER_005
user_id,,,,,
USER_001,1.000000,0.000000,0.210775,0.022161,0.0
USER_002,0.000000,1.000000,0.357440,0.265279,0.0
USER_003,0.210775,0.357440,1.000000,0.331595,0.0
USER_004,0.022161,0.265279,0.331595,1.000000,0.0
USER_005,0.000000,0.000000,0.000000,0.000000,1.0


In [146]:
print("\nMATRIX VALIDATION")
print("=" * 50)

print("Training users:", user_job_matrix.shape[0])
print("Training jobs:", user_job_matrix.shape[1])
print("Non-zero interactions:", np.count_nonzero(user_job_matrix.values))

total_possible = user_job_matrix.size

sparsity = (
    1 - np.count_nonzero(user_job_matrix.values) / total_possible
) * 100

print("Matrix sparsity:", round(sparsity, 2), "%")

print("Missing users:", 
      len(set(train_interactions['user_id']) - set(user_job_matrix.index)))

print("Similarity matrix shape:", user_similarity_df.shape)


MATRIX VALIDATION
Training users: 4000
Training jobs: 1020
Non-zero interactions: 64198
Matrix sparsity: 98.43 %
Missing users: 0
Similarity matrix shape: (4000, 4000)


**training interaction graph**

In [147]:

# Build a training-only user-job bipartite graph

G = nx.Graph()

# Add all user nodes
for user_id in train_interactions['user_id'].unique():
    G.add_node(
        user_id,
        node_type='user'
    )

# Add all job nodes
for job_id in jobs['id']:
    G.add_node(
        f"JOB_{job_id}",
        node_type='job'
    )

# Add edges using training interactions only
for _, row in train_interactions.iterrows():

    user_id = row['user_id']
    job_node = f"JOB_{int(row['job_id'])}"

    G.add_edge(
        user_id,
        job_node,
        weight=row['rating'],
        interaction_type=row['interaction_type']
    )

# Display graph statistics
print("TRAINING INTERACTION GRAPH")
print("=" * 50)

print("Total nodes:", G.number_of_nodes())
print("Total edges:", G.number_of_edges())

print("\nNode types:")
print({
    'users': sum(
        1 for _, data in G.nodes(data=True)
        if data['node_type'] == 'user'
    ),
    'jobs': sum(
        1 for _, data in G.nodes(data=True)
        if data['node_type'] == 'job'
    )
})

print("\nExpected nodes:", 4000 + 1020)
print("Expected edges:", len(train_interactions))

print("\nGraph validation:")
print("Correct node count:", G.number_of_nodes() == 5020)
print("Correct edge count:", G.number_of_edges() == 64198)


TRAINING INTERACTION GRAPH
Total nodes: 5020
Total edges: 64198

Node types:
{'users': 4000, 'jobs': 1020}

Expected nodes: 5020
Expected edges: 64198

Graph validation:
Correct node count: True
Correct edge count: True


In [148]:

# ==========================================
# 8. PREPARE TEST RELEVANCE DATA
# ==========================================

# Group testing interactions by user
test_relevant_jobs = (
    test_interactions.groupby('user_id')['job_id']
    .apply(set)
    .to_dict()
)

# Display test relevance statistics
print("TEST RELEVANCE DATA")
print("=" * 50)

print("Users with relevant test jobs:", len(test_relevant_jobs))

print(
    "Total relevant test interactions:",
    sum(len(job_ids) for job_ids in test_relevant_jobs.values())
)

print("\nRelevant jobs for sample users:")

for user_id in list(test_relevant_jobs.keys())[:5]:
    print(user_id, ":", len(test_relevant_jobs[user_id]), "jobs")


TEST RELEVANCE DATA
Users with relevant test jobs: 4000
Total relevant test interactions: 16026

Relevant jobs for sample users:
USER_001 : 5 jobs
USER_002 : 2 jobs
USER_003 : 5 jobs
USER_004 : 2 jobs
USER_005 : 3 jobs


In [149]:

# ==========================================
# 9. DEFINE EVALUATION METRICS
# ==========================================

def precision_at_k(recommended_jobs, relevant_jobs, k=5):
    recommended = recommended_jobs[:k]

    if not recommended:
        return 0.0

    relevant_recommended = len(
        set(recommended) & relevant_jobs
    )

    return relevant_recommended / k


def recall_at_k(recommended_jobs, relevant_jobs, k=5):
    if not relevant_jobs:
        return 0.0

    recommended = recommended_jobs[:k]

    relevant_recommended = len(
        set(recommended) & relevant_jobs
    )

    return relevant_recommended / len(relevant_jobs)


def f1_at_k(precision, recall):
    if precision + recall == 0:
        return 0.0

    return 2 * (precision * recall) / (precision + recall)


print("Evaluation metrics defined successfully!")


Evaluation metrics defined successfully!


*Create the Personalized PageRank function*

In [150]:
def get_graph_scores(user_id, graph, train_interactions, alpha=0.85):

    # Check that the user exists in the graph
    if user_id not in graph:
        return {}

    # Create personalization vector
    personalization = {
        node: 0.0 for node in graph.nodes()
    }

    personalization[user_id] = 1.0

    # Calculate Personalized PageRank
    pagerank_scores = nx.pagerank(
        graph,
        alpha=alpha,
        personalization=personalization,
        weight='weight'
    )

    # Get jobs already interacted with by this user
    seen_jobs = set(
        train_interactions[
            train_interactions['user_id'] == user_id
        ]['job_id']
    )

    # Keep only unseen job scores
    job_scores = {}

    for node, score in pagerank_scores.items():

        if node.startswith("JOB_"):

            job_id = int(node.replace("JOB_", ""))

            if job_id not in seen_jobs:
                job_scores[job_id] = score

    return job_scores

In [151]:
# Generate graph-based scores
graph_scores = get_graph_scores(
    user_id='USER_001',
    graph=G,
    train_interactions=train_interactions
)

# Display top 10 graph recommendations
top_graph_jobs = sorted(
    graph_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

graph_recommendations = pd.DataFrame(
    top_graph_jobs,
    columns=['job_id', 'graph_score']
)

graph_recommendations = graph_recommendations.merge(
    jobs[['id', 'title', 'company', 'location']],
    left_on='job_id',
    right_on='id',
    how='left'
)

graph_recommendations = graph_recommendations.drop(
    columns=['id']
)

display(graph_recommendations)

,job_id,graph_score,title,company,location
0,346,0.006460,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Dallas, TX"
1,378,0.006281,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Fulton, MD"
2,411,0.005939,Software Engineer Sr - Cloud Engineer,PNC,"Farmers Branch, TX"
3,28,0.005784,Software Developer Intern,IBM,"Tucson, AZ"
4,143,0.005616,2026 Dark Wolf Summer Internship,Dark Wolf,"Herndon, VA"
5,282,0.004534,"Software Engineer, Ground Software",Muon Space,"San Jose, CA"
6,237,0.004453,Software Developer Co-Op Fall 2026,IBM,"Durham, NC"
7,208,0.004280,"Manager, AI Platform Engineering (Remote)",The Home Depot,"Atlanta, GA"
8,436,0.003890,Software Engineer Intern,Leidos,"Reston, VA"
9,335,0.003777,Software Engineer,Winaxis LLC,"Lewes, DE"


*Content-Based scoring*

In [152]:
def get_content_scores(user_id, train_interactions):

    # Get this user's training interactions
    user_history = train_interactions[
        train_interactions['user_id'] == user_id
    ]

    # Return empty dictionary if user has no history
    if user_history.empty:
        return {}

    # Get job IDs and their ratings
    interacted_jobs = user_history['job_id'].values
    ratings = user_history['rating'].values

    # Convert job IDs into similarity matrix indices
    job_indices = [
        job_id_to_index[job_id]
        for job_id in interacted_jobs
    ]

    # Calculate weighted similarity scores
    scores = np.average(
        combined_similarity[:, job_indices],
        axis=1,
        weights=ratings
    )

    # Exclude jobs already interacted with
    seen_jobs = set(interacted_jobs)

    content_scores = {}

    for index, score in enumerate(scores):

        job_id = int(jobs.iloc[index]['id'])

        if job_id not in seen_jobs:
            content_scores[job_id] = score

    return content_scores

In [153]:
content_scores = get_content_scores(
    user_id='USER_001',
    train_interactions=train_interactions
)

# Get top 10 recommendations
top_content_jobs = sorted(
    content_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

content_recommendations = pd.DataFrame(
    top_content_jobs,
    columns=['job_id', 'content_score']
)

content_recommendations = content_recommendations.merge(
    jobs[['id', 'title', 'company', 'location']],
    left_on='job_id',
    right_on='id',
    how='left'
)

content_recommendations = content_recommendations.drop(
    columns=['id']
)

display(content_recommendations)

,job_id,content_score,title,company,location
0,346,0.448487,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Dallas, TX"
1,378,0.448487,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Fulton, MD"
2,282,0.428649,"Software Engineer, Ground Software",Muon Space,"San Jose, CA"
3,436,0.427789,Software Engineer Intern,Leidos,"Reston, VA"
4,28,0.419966,Software Developer Intern,IBM,"Tucson, AZ"
5,840,0.415977,Java Developer,The Agora Companies,"Baltimore, MD"
6,608,0.415219,Lead Java Backend Developer,The Value Maximizer,"Pennsylvania, United States"
7,403,0.392438,Software Engineer Associate - Level 1,Lockheed Martin,"Fort Worth, TX"
8,847,0.389058,"Java Programmer/Manager (Spring.IO, Java, MySQL, AWS)",ALTUS Commercial Receivables,"Metairie, LA"
9,292,0.386076,Senior Software Engineer,Optum,"Irvine, CA"


*Collaborative Filtering scoring*

In [154]:
def get_collaborative_scores(user_id, user_job_matrix,
                             user_similarity_df, train_interactions,
                             top_k=10):

    # Check whether the user exists
    if user_id not in user_job_matrix.index:
        return {}

    # Get similarity scores for this user
    similarities = user_similarity_df.loc[user_id].drop(user_id)

    # Select the top K most similar users
    neighbors = similarities.nlargest(top_k)

    # Keep only users with positive similarity
    neighbors = neighbors[neighbors > 0]

    if neighbors.empty:
        return {}

    # Get ratings from the selected neighbours
    neighbor_ratings = user_job_matrix.loc[neighbors.index]

    # Calculate weighted average ratings for each job
    weighted_sum = neighbor_ratings.mul(
        neighbors, axis=0
    ).sum(axis=0)

    weighted_denominator = (
        neighbor_ratings.ne(0)
        .mul(neighbors, axis=0)
        .sum(axis=0)
    )

    collaborative_scores = (
        weighted_sum / weighted_denominator.replace(0, np.nan)
    ).dropna()

    # Exclude jobs already interacted with by the target user
    seen_jobs = set(
        train_interactions[
            train_interactions['user_id'] == user_id
        ]['job_id']
    )

    collaborative_scores = collaborative_scores[
        ~collaborative_scores.index.isin(seen_jobs)
    ]

    return collaborative_scores.to_dict()

In [155]:
collaborative_scores = get_collaborative_scores(
    user_id='USER_001',
    user_job_matrix=user_job_matrix,
    user_similarity_df=user_similarity_df,
    train_interactions=train_interactions,
    top_k=10
)

# Get top 10 recommendations
top_cf_jobs = sorted(
    collaborative_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

cf_recommendations = pd.DataFrame(
    top_cf_jobs,
    columns=['job_id', 'collaborative_score']
)

cf_recommendations['job_id'] = cf_recommendations['job_id'].astype(int)

cf_recommendations = cf_recommendations.merge(
    jobs[['id', 'title', 'company', 'location']],
    left_on='job_id',
    right_on='id',
    how='left'
)

cf_recommendations = cf_recommendations.drop(
    columns=['id']
)

display(cf_recommendations)

,job_id,collaborative_score,title,company,location
0,200,5.0,"Solutions Architect, AI and ML",NVIDIA,"Seattle, WA"
1,561,5.0,software engineering intern (backend / robotics infrastructure）,mundane,"Palo Alto, CA"
2,282,3.0,"Software Engineer, Ground Software",Muon Space,"San Jose, CA"
3,644,3.0,React Software Developer (Part-Time),Aretum,"McLean, VA"
4,126,3.0,AI Enabling Specialist - Finance,NVIDIA,"Santa Clara, CA"
5,270,3.0,Software Engineer I (Full Time) - United States,Cisco,"San Jose, CA"
6,283,3.0,Senior Software Engineer,Optum,"Bridgeton, MO"
7,387,3.0,Software Tools Automation Intern,Nihon Kohden Digital Health Solutions,"Irvine, CA"
8,460,3.0,Back End Developer Intern 2026,IBM,"Lowell, MA"
9,594,3.0,"Software Engineer, Bill Pay & Procurement",Ramp,"New York, NY"


In [156]:
# Test all three recommendation models

user_id = 'USER_001'

content_scores = get_content_scores(
    user_id, train_interactions
)

collaborative_scores = get_collaborative_scores(
    user_id,
    user_job_matrix,
    user_similarity_df,
    train_interactions,
    top_k=10
)

graph_scores = get_graph_scores(
    user_id,
    G,
    train_interactions
)

print("RECOMMENDATION SCORE VALIDATION")
print("=" * 45)

print("Content-based recommendations:", len(content_scores))
print("Collaborative recommendations:", len(collaborative_scores))
print("Graph-based recommendations:", len(graph_scores))

RECOMMENDATION SCORE VALIDATION
Content-based recommendations: 1001
Collaborative recommendations: 58
Graph-based recommendations: 1001


*Hybrid Recommendation*

In [157]:
def normalize_score_dict(scores):

    if not scores:
        return {}

    values = np.array(list(scores.values()), dtype=float)

    min_score = values.min()
    max_score = values.max()

    # Equal scores provide no ranking information
    if max_score == min_score:
        return {job_id: 0.0 for job_id in scores}

    normalized = (values - min_score) / (max_score - min_score)

    return dict(zip(scores.keys(), normalized))

In [158]:
def get_hybrid_scores(user_id, train_interactions):

    # Get scores from all three recommendation components
    content_scores = get_content_scores(
        user_id, train_interactions
    )

    cf_scores = get_collaborative_scores(
        user_id,
        user_job_matrix,
        user_similarity_df,
        train_interactions,
        top_k=10
    )

    graph_scores = get_graph_scores(
        user_id,
        G,
        train_interactions
    )

    # Normalize component scores independently
    content_normalized = normalize_score_dict(content_scores)
    cf_normalized = normalize_score_dict(cf_scores)
    graph_normalized = normalize_score_dict(graph_scores)

    # Combine all candidate jobs
    candidate_jobs = (
        set(content_normalized) |
        set(cf_normalized) |
        set(graph_normalized)
    )

    hybrid_scores = {}

    for job_id in candidate_jobs:

        content_score = content_normalized.get(job_id, 0)
        cf_score = cf_normalized.get(job_id, 0)
        graph_score = graph_normalized.get(job_id, 0)

        hybrid_score = (
            0.50 * content_score +
            0.30 * cf_score +
            0.20 * graph_score
        )

        hybrid_scores[job_id] = hybrid_score

    return hybrid_scores

In [159]:
hybrid_scores = get_hybrid_scores(
    user_id='USER_001',
    train_interactions=train_interactions
)

top_hybrid_jobs = sorted(
    hybrid_scores.items(),
    key=lambda x: x[1],
    reverse=True
)[:10]

hybrid_recommendations = pd.DataFrame(
    top_hybrid_jobs,
    columns=['job_id', 'hybrid_score']
)

hybrid_recommendations = hybrid_recommendations.merge(
    jobs[['id', 'title', 'company', 'location']],
    left_on='job_id',
    right_on='id',
    how='left'
)

hybrid_recommendations = hybrid_recommendations.drop(
    columns=['id']
)

display(hybrid_recommendations)

,job_id,hybrid_score,title,company,location
0,346,0.801107,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Dallas, TX"
1,282,0.768067,"Software Engineer, Ground Software",Muon Space,"San Jose, CA"
2,28,0.758143,Software Developer Intern,IBM,"Tucson, AZ"
3,378,0.742763,Software Engineer Full Stack & Application Development I (Full Time) – United States,Cisco,"Fulton, MD"
4,411,0.719524,Software Engineer Sr - Cloud Engineer,PNC,"Farmers Branch, TX"
5,436,0.695648,Software Engineer Intern,Leidos,"Reston, VA"
6,561,0.675818,software engineering intern (backend / robotics infrastructure）,mundane,"Palo Alto, CA"
7,335,0.648384,Software Engineer,Winaxis LLC,"Lewes, DE"
8,208,0.619561,"Manager, AI Platform Engineering (Remote)",The Home Depot,"Atlanta, GA"
9,143,0.595855,2026 Dark Wolf Summer Internship,Dark Wolf,"Herndon, VA"


In [160]:
# Relevant test interactions: saves and applications
relevant_test = test_interactions[
    test_interactions['rating'] >= 3
].copy()

# Group relevant job IDs by user
test_relevant_jobs = (
    relevant_test
    .groupby('user_id')['job_id']
    .apply(set)
    .to_dict()
)

# Users who have at least one relevant test interaction
eligible_users = list(test_relevant_jobs.keys())

print("TEST DATA FOR EVALUATION")
print("=" * 50)

print("Total test interactions:", len(test_interactions))
print("Relevant test interactions:", len(relevant_test))
print("Users in test set:", test_interactions['user_id'].nunique())
print("Users with relevant test jobs:", len(eligible_users))
print("Users without relevant test jobs:",
      test_interactions['user_id'].nunique() - len(eligible_users))

print("\nRelevant interaction distribution:")
print(relevant_test['interaction_type'].value_counts())

print("\nRelevant jobs per eligible user:")
print(relevant_test.groupby('user_id')['job_id'].count().describe())

TEST DATA FOR EVALUATION
Total test interactions: 16026
Relevant test interactions: 9551
Users in test set: 4000
Users with relevant test jobs: 3609
Users without relevant test jobs: 391

Relevant interaction distribution:
interaction_type
save     6791
apply    2760
Name: count, dtype: int64

Relevant jobs per eligible user:
count    3609.000000
mean        2.646439
std         1.357297
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max         7.000000
Name: job_id, dtype: float64


*Ground-truth data*

In [161]:

# ==========================================
# GROUND TRUTH PREPARATION AND VALIDATION
# ==========================================

# Select relevant test interactions (saves and applications)
relevant_test = test_interactions[
    test_interactions['rating'] >= 3
].copy()

# Create ground truth: relevant jobs for each user
ground_truth = (
    relevant_test.groupby('user_id')['job_id']
    .apply(set)
    .to_dict()
)

# Users eligible for evaluation
evaluation_users = list(ground_truth.keys())

print("GROUND TRUTH VALIDATION")
print("=" * 50)

print("Total test interactions:", len(test_interactions))
print("Relevant test interactions:", len(relevant_test))
print("Eligible users:", len(evaluation_users))

print(
    "Users without relevant test jobs:",
    len(test_interactions['user_id'].unique()) - len(evaluation_users)
)

print("\nRelevant interaction distribution:")
print(relevant_test['interaction_type'].value_counts())

print("\nRelevant jobs per user:")
print(relevant_test.groupby('user_id')['job_id'].nunique().describe())

print("\nExample ground truth:")
for user_id in evaluation_users[:5]:
    print(user_id, ":", ground_truth[user_id])

print("\nGROUND TRUTH CHECKS")
print("=" * 50)

print(
    "Correct relevant interaction count:",
    len(relevant_test) == 9551
)

print(
    "Correct eligible user count:",
    len(evaluation_users) == 3609
)

print(
    "All eligible users have relevant jobs:",
    all(len(job_ids) > 0 for job_ids in ground_truth.values())
)

print(
    "All ground truth jobs exist in dataset:",
    set().union(*ground_truth.values()).issubset(set(jobs['id']))
)


GROUND TRUTH VALIDATION
Total test interactions: 16026
Relevant test interactions: 9551
Eligible users: 3609
Users without relevant test jobs: 391

Relevant interaction distribution:
interaction_type
save     6791
apply    2760
Name: count, dtype: int64

Relevant jobs per user:
count    3609.000000
mean        2.646439
std         1.357297
min         1.000000
25%         1.000000
50%         2.000000
75%         4.000000
max         7.000000
Name: job_id, dtype: float64

Example ground truth:
USER_001 : {378, 28}
USER_002 : {568}
USER_003 : {28}
USER_004 : {356}
USER_005 : {331, 421}

GROUND TRUTH CHECKS
Correct relevant interaction count: True
Correct eligible user count: True
All eligible users have relevant jobs: True
All ground truth jobs exist in dataset: True


**Metric function**

In [162]:
def precision_recall_f1_at_k(recommended_jobs, relevant_jobs, k=5):

    # Take the top K recommendations
    recommended_jobs = recommended_jobs[:k]

    # Convert to sets for comparison
    recommended_set = set(recommended_jobs)
    relevant_set = set(relevant_jobs)

    # Count relevant recommendations
    hits = len(recommended_set.intersection(relevant_set))

    # Calculate precision
    precision = hits / k

    # Calculate recall
    recall = (
        hits / len(relevant_set)
        if len(relevant_set) > 0
        else 0
    )

    # Calculate F1-score
    f1 = (
        2 * precision * recall / (precision + recall)
        if (precision + recall) > 0
        else 0
    )

    return precision, recall, f1

In [163]:
# Example test
example_recommendations = [101, 102, 103, 104, 105]
example_relevant_jobs = {102, 104, 106}

p, r, f1 = precision_recall_f1_at_k(
    example_recommendations,
    example_relevant_jobs,
    k=5
)

print("METRIC FUNCTION TEST")
print("=" * 40)
print("Precision@5:", round(p, 4))
print("Recall@5:", round(r, 4))
print("F1@5:", round(f1, 4))

METRIC FUNCTION TEST
Precision@5: 0.4
Recall@5: 0.6667
F1@5: 0.5


*Evaluate all four recommendation algorithms*

In [164]:
# ==========================================
# EVALUATE ALL FOUR RECOMMENDATION MODELS
# Using a sample of 500 users
# ==========================================

# Step 1: Select a reproducible sample of 500 users
sample_users = (
    pd.Series(evaluation_users)
    .sample(n=1000, random_state=42)
    .tolist()
)

print("Users selected for evaluation:", len(sample_users))


# Step 2: Store evaluation results
evaluation_results = []


# Step 3: Evaluate each selected user
for i, user_id in enumerate(sample_users, start=1):

    relevant_jobs = ground_truth[user_id]

    # Generate content-based scores
    content_scores = get_content_scores(
        user_id,
        train_interactions
    )

    # Generate collaborative filtering scores
    cf_scores = get_collaborative_scores(
        user_id,
        user_job_matrix,
        user_similarity_df,
        train_interactions,
        top_k=10
    )

    # Generate graph-based scores
    graph_scores = get_graph_scores(
        user_id,
        G,
        train_interactions
    )

    # Normalize component scores
    content_normalized = normalize_score_dict(content_scores)
    cf_normalized = normalize_score_dict(cf_scores)
    graph_normalized = normalize_score_dict(graph_scores)

    # Combine candidate jobs from all three models
    candidate_jobs = (
        set(content_normalized) |
        set(cf_normalized) |
        set(graph_normalized)
    )

    # Calculate hybrid scores
    hybrid_scores = {}

    for job_id in candidate_jobs:
        hybrid_scores[job_id] = (
            0.50 * content_normalized.get(job_id, 0) +
            0.30 * cf_normalized.get(job_id, 0) +
            0.20 * graph_normalized.get(job_id, 0)
        )

    # Collect recommendations from all four models
    recommendations = {
        'Content-Based': content_scores,
        'Collaborative': cf_scores,
        'Graph-Based': graph_scores,
        'Hybrid': hybrid_scores
    }

    # Evaluate each recommendation model
    for model_name, scores in recommendations.items():

        ranked_jobs = [
            job_id
            for job_id, score in sorted(
                scores.items(),
                key=lambda x: x[1],
                reverse=True
            )[:5]
        ]

        precision, recall, f1 = precision_recall_f1_at_k(
            ranked_jobs,
            relevant_jobs,
            k=5
        )

        evaluation_results.append({
            'user_id': user_id,
            'model': model_name,
            'precision@5': precision,
            'recall@5': recall,
            'f1@5': f1,
            'recommendation_count': len(ranked_jobs),
            'has_recommendations': len(ranked_jobs) > 0
        })

    # Display progress after every 25 users
    if i % 25 == 0 or i == len(sample_users):
        print(f"Evaluated {i}/{len(sample_users)} users")


# Step 4: Convert results into a DataFrame
evaluation_df = pd.DataFrame(evaluation_results)


# Step 5: Display evaluation summary
print("\nEVALUATION COMPLETED")
print("=" * 50)

print("Users evaluated:", len(sample_users))
print("Models evaluated:", evaluation_df['model'].nunique())
print("Total evaluation records:", len(evaluation_df))

display(evaluation_df.head(12))


Users selected for evaluation: 1000
Evaluated 25/1000 users
Evaluated 50/1000 users
Evaluated 75/1000 users
Evaluated 100/1000 users
Evaluated 125/1000 users
Evaluated 150/1000 users
Evaluated 175/1000 users
Evaluated 200/1000 users
Evaluated 225/1000 users
Evaluated 250/1000 users
Evaluated 275/1000 users
Evaluated 300/1000 users
Evaluated 325/1000 users
Evaluated 350/1000 users
Evaluated 375/1000 users
Evaluated 400/1000 users
Evaluated 425/1000 users
Evaluated 450/1000 users
Evaluated 475/1000 users
Evaluated 500/1000 users
Evaluated 525/1000 users
Evaluated 550/1000 users
Evaluated 575/1000 users
Evaluated 600/1000 users
Evaluated 625/1000 users
Evaluated 650/1000 users
Evaluated 675/1000 users
Evaluated 700/1000 users
Evaluated 725/1000 users
Evaluated 750/1000 users
Evaluated 775/1000 users
Evaluated 800/1000 users
Evaluated 825/1000 users
Evaluated 850/1000 users
Evaluated 875/1000 users
Evaluated 900/1000 users
Evaluated 925/1000 users
Evaluated 950/1000 users
Evaluated 975/100

,user_id,model,precision@5,recall@5,f1@5,recommendation_count,has_recommendations
0,USER_3978,Content-Based,0.0,0.00,0.000000,5,True
1,USER_3978,Collaborative,0.0,0.00,0.000000,5,True
2,USER_3978,Graph-Based,0.0,0.00,0.000000,5,True
3,USER_3978,Hybrid,0.0,0.00,0.000000,5,True
4,USER_3132,Content-Based,0.2,0.25,0.222222,5,True
5,USER_3132,Collaborative,0.2,0.25,0.222222,5,True
6,USER_3132,Graph-Based,0.0,0.00,0.000000,5,True
7,USER_3132,Hybrid,0.0,0.00,0.000000,5,True
8,USER_2962,Content-Based,0.0,0.00,0.000000,5,True
9,USER_2962,Collaborative,0.4,0.40,0.400000,5,True


In [167]:
# Aggregate evaluation metrics by recommendation model
model_performance = (
    evaluation_df.groupby('model')
    .agg(
        precision_at_5=('precision@5', 'mean'),
        recall_at_5=('recall@5', 'mean'),
        f1_at_5=('f1@5', 'mean'),
        coverage=('has_recommendations', 'mean')
    )
    .reset_index()
)

# Convert coverage to percentage
model_performance['coverage'] *= 100

# Sort by F1-score
model_performance = model_performance.sort_values(
    by='f1_at_5',
    ascending=False
)

# Round results
model_performance = model_performance.round(4)

print("MODEL PERFORMANCE COMPARISON")
print("=" * 60)

display(model_performance)

MODEL PERFORMANCE COMPARISON


,model,precision_at_5,recall_at_5,f1_at_5,coverage
2,Graph-Based,0.2280,0.4396,0.2817,100.0
3,Hybrid,0.2046,0.3934,0.2522,100.0
1,Content-Based,0.1236,0.2348,0.1525,100.0
0,Collaborative,0.0788,0.1485,0.0965,100.0


In [168]:
import os

# Create processed data directory if it doesn't exist
os.makedirs("../data/processed", exist_ok=True)

# Save individual evaluation results
evaluation_df.to_csv(
    "../data/processed/model_evaluation_results.csv",
    index=False
)

# Save aggregated model performance
model_performance.to_csv(
    "../data/processed/model_performance_comparison.csv",
    index=False
)

print("Evaluation results saved successfully!")

Evaluation results saved successfully!
